# 2D Wavelet Decomposition

In the previous chapter you built a 1D wavelet transform from scratch and saw how it concentrates signal energy into a small number of large coefficients. Real MR images are two-dimensional, so we need to extend the decomposition to 2D. The good news: the 2D transform is simply the 1D transform applied **separably** — first along rows, then along columns.

This chapter is a brief visual overview rather than a deep dive. The goal is to build enough intuition to understand how wavelet-based **compression** works — the same principle behind JPEG 2000. For compressed sensing in MRI, sparsity in the wavelet domain is exactly what allows reconstruction from undersampled k-space.

In [1]:
import pathlib
from IPython.display import HTML

_root = pathlib.Path.cwd()
if not (_root / '_static').exists():
    _root = _root.parent

def _read_js(*names):
    return '\n'.join((_root / '_static' / 'js' / n).read_text() for n in names)

def _embed(controls_html, js_code, wait_for=None):
    if wait_for:
        init_guard = f'''if (typeof window.{wait_for} === "undefined") {{ setTimeout(initBlock, 100); return; }}'''
    else:
        init_guard = ''
    return HTML(f'''
<style>
.controls-panel {{
  background: #f9f6f2; border: 1px solid #e0dcd6; border-radius: 4px;
  padding: 1rem 1.2rem; margin: 1.5rem 0; display: flex; flex-wrap: wrap;
  gap: 0.8rem 1.5rem; align-items: end;
}}
.controls-panel label {{
  display: flex; flex-direction: column; font-size: 0.85rem; color: #555; gap: 0.2rem;
}}
.controls-panel input[type="number"], .controls-panel select {{
  font-size: 0.9rem; padding: 0.25rem 0.4rem; border: 1px solid #ccc;
  border-radius: 3px; width: 7rem;
}}
.controls-panel select {{ width: 8rem; }}
</style>
{controls_html}
<script>
(function() {{
  function initBlock() {{
    {init_guard}
    {js_code}
  }}
  if (typeof Plotly !== "undefined") {{ initBlock(); }}
  else {{
    var s = document.createElement("script");
    s.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    s.onload = initBlock;
    document.head.appendChild(s);
  }}
}})();
</script>
''')

---

## Load an image

Select an image file below. It will be converted to grayscale and downscaled so the largest dimension is 512 pixels.

In [2]:
_embed(
    '''
    <div class="controls-panel">
      <label>Choose image
        <input type="file" id="img-file-input" accept="image/*" style="width:auto;">
      </label>
    </div>
    <p id="img-info" style="font-size:0.85rem; color:#555; margin:0.3rem 0;"></p>
    <div id="img-preview-plot" style="width:100%; height:500px;"></div>
    ''',
    _read_js('img_loader.js')
)

---

## The filters

We use the same Haar filters from Chapter 1:

$$L = \left[\frac{1}{\sqrt{2}},\; \frac{1}{\sqrt{2}}\right] \qquad H = \left[\frac{1}{\sqrt{2}},\; -\frac{1}{\sqrt{2}}\right]$$

In 2D, the transform is **separable**: we apply the 1D filters along rows first, then along columns. This produces four sub-images per level, each corresponding to a 2×2 kernel that is the outer product of the row and column filters:

In [3]:
_embed(
    '''
    <div id="filters-2d-plot" style="width:100%; height:250px;"></div>
    <div id="filters-2d-kernels-plot" style="width:100%; height:200px;"></div>
    ''',
    _read_js('filters_2d.js')
)

---

## Pass 1: Filter along rows

First, we slide the $L$ and $H$ filters across each row of the image, then downsample by 2. This produces two intermediate images: **L→** (low-pass along rows, half the width) and **H→** (high-pass along rows, half the width).

Use the **row** slider to select which row is being filtered, and the **position** slider to watch the filter slide across that row.

In [4]:
_embed(
    '''
    <div style="display:flex; gap:1.5rem; flex-wrap:wrap; margin:1rem 0;">
      <label style="font-size:0.85rem; color:#555; flex:1;">Row
        <input type="range" id="dec2d-p1-row" min="0" max="255" value="0" style="width:100%;">
      </label>
      <label style="font-size:0.85rem; color:#555; flex:1;">Filter position
        <input type="range" id="dec2d-p1-pos" min="0" max="255" value="0" style="width:100%;">
      </label>
    </div>
    <p id="dec2d-p1-info" style="font-size:0.85rem; color:#555; margin:0.3rem 0;"></p>
    <div id="dec2d-p1-plot" style="width:100%; height:400px;"></div>
    ''',
    _read_js('decompose_2d.js'),
    wait_for='ImageData2D'
)

---

## Pass 2: Filter along columns

Now we take the two intermediate images from Pass 1 and slide the $L$ and $H$ filters down each column, then downsample by 2. This produces the four sub-images:

- **LL** — $L$ along rows, then $L$ along columns (approximation)
- **LH** — $L$ along rows, then $H$ along columns (horizontal detail)
- **HL** — $H$ along rows, then $L$ along columns (vertical detail)
- **HH** — $H$ along rows, then $H$ along columns (diagonal detail)

Use the **column** slider to select which column is being filtered, and the **position** slider to watch the filter slide down.

In [5]:
_embed(
    '''
    <div style="display:flex; gap:1.5rem; flex-wrap:wrap; margin:1rem 0;">
      <label style="font-size:0.85rem; color:#555; flex:1;">Column
        <input type="range" id="dec2d-p2-col" min="0" max="127" value="0" style="width:100%;">
      </label>
      <label style="font-size:0.85rem; color:#555; flex:1;">Filter position
        <input type="range" id="dec2d-p2-pos" min="0" max="255" value="0" style="width:100%;">
      </label>
    </div>
    <p id="dec2d-p2-info" style="font-size:0.85rem; color:#555; margin:0.3rem 0;"></p>
    <div id="dec2d-p2-plot" style="width:100%; height:600px;"></div>
    ''',
    '',
    wait_for='DecompIntermediate'
)

---

## The cascade

Just as in 1D, we can repeat the decomposition on the LL sub-image to separate progressively coarser features. At each level, LL gets subdivided into four new sub-images while the detail sub-images (LH, HL, HH) are kept. Use the slider to add more levels and watch the top-left corner get recursively decomposed.


In [6]:
# Cascade: multi-level 2D decomposition
_embed(
    '''
    <div class="controls-panel">
      <label>Decomposition levels <input type="number" id="cascade2d-levels" value="1" step="1" min="1" max="6"></label>
    </div>
    <p id="cascade2d-info" style="font-size:0.85rem; color:#555; margin:0.3rem 0;"></p>
    <div id="cascade2d-plot" style="width:100%; height:600px;"></div>
    <div id="cascade2d-compare-plot" style="width:100%; height:400px; margin-top:1rem;"></div>
    ''',
    _read_js('cascade_2d.js'),
    wait_for='ImageData2D'
)


---

## Thresholding and compression

The wavelet transform itself does not compress — it just reorganizes the data. The actual compression comes from **thresholding**: zeroing out small detail coefficients, then storing only the non-zero ones.

Use the slider below to set a threshold. Any detail coefficient with absolute value below the threshold is set to zero. The remaining non-zero coefficients are stored as a sparse list (row index + column index + value = 8 bytes each), and the image is reconstructed from this sparse representation.

Watch the tradeoff: higher threshold → fewer coefficients → smaller storage → more visible artifacts in the reconstruction.

This is the core idea behind wavelet-based image compression (JPEG 2000): most of an image's energy concentrates in a small number of large coefficients — the image is **sparse** in the wavelet domain. In compressed sensing for MRI, we exploit this same sparsity but from the other direction: instead of acquiring all the data and then throwing away small coefficients, we acquire fewer k-space samples from the start and use the knowledge that the image is wavelet-sparse to fill in the gaps.

In [7]:
_embed(
    '''
    <div style="margin:1rem 0;">
      <label style="font-size:0.85rem; color:#555;">Threshold
        <input type="range" id="thr2d-threshold" min="0" max="10000" value="0" step="100" style="width:100%;">
      </label>
    </div>
    <p id="thr2d-info" style="font-size:0.85rem; color:#555; margin:0.3rem 0;"></p>
    <div id="thr2d-plot" style="width:100%; height:450px;"></div>
    ''',
    _read_js('threshold_2d.js'),
    wait_for='ImageData2D'
)
